In [ ]:
# import data handling tools
import os
import numpy as np
import pandas as pd
import seaborn as sns
sns.set_style('darkgrid')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

from PIL import Image

# import Deep learning Libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam, Adamax
from tensorflow.keras.metrics import categorical_crossentropy
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Activation, Dropout, BatchNormalization
from tensorflow.keras import regularizers
import cv2
# Ignore Warnings
import warnings
warnings.filterwarnings("ignore")

print ('modules loaded')

In [ ]:
dataset_path = "dataset"

train_path = os.path.join(dataset_path, "train")
val_path = os.path.join(dataset_path, "val")
test_path = os.path.join(dataset_path, "test")

In [ ]:
classes = os.listdir(train_path)
print("Classes:", classes)

In [ ]:
def count_images(folder):
    counts = {}
    for cls in os.listdir(folder):
        cls_path = os.path.join(folder, cls)
        counts[cls] = len(os.listdir(cls_path))
    return counts

train_counts = count_images(train_path)
val_counts = count_images(val_path)
test_counts = count_images(test_path)

print("Train:", train_counts)
print("Validation:", val_counts)
print("Test:", test_counts)

In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(x=list(train_counts.keys()), y=list(train_counts.values()))
plt.title("Training Data Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Images")
plt.xticks(rotation=30)
plt.show()

In [ ]:
records = []

for split in ['train','val','test']:
    split_path = os.path.join(dataset_path, split)

    for label in os.listdir(split_path):
        class_path = os.path.join(split_path, label)

        for img in os.listdir(class_path):
            records.append({
                "split": split,
                "label": label,
                "filepath": os.path.join(class_path, img)
            })

eda_df = pd.DataFrame(records)

print("Total images:", len(eda_df))
eda_df.head()

In [ ]:
sizes = []

sample_paths = eda_df['filepath'].sample(200)

for path in sample_paths:
    img = Image.open(path)
    sizes.append(img.size)

size_df = pd.DataFrame(sizes, columns=['width','height'])

print(size_df.describe())

In [ ]:
sample_img = np.array(Image.open(eda_df['filepath'].iloc[0]))

print("Min pixel value:", sample_img.min())
print("Max pixel value:", sample_img.max())

In [ ]:
samples_per_class = 3

for label in classes:
    class_path = os.path.join(train_path, label)

    images = os.listdir(class_path)[:samples_per_class]

    plt.figure(figsize=(10,3))

    for i, img_name in enumerate(images):
        img = cv2.imread(os.path.join(class_path, img_name))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        plt.subplot(1,3,i+1)
        plt.imshow(img)
        plt.title(label)
        plt.axis("off")

    plt.show()

In [ ]:
broken = []

for path in eda_df['filepath']:
    try:
        Image.open(path)
    except:
        broken.append(path)

print("Broken images:", len(broken))

## Extended EDA Visualizations (25+)
These cells add a broad set of charts (bars, pies, heatmaps, histograms, box/violin plots, and scatter/hexbin) built from a sampled subset of the dataset for speed. Run them after the earlier setup cells.

In [ ]:
# Build a sampled stats dataframe for efficient plotting
from tqdm import tqdm
tqdm.pandas()

# sample a manageable subset to compute stats
stat_sample = eda_df.sample(min(len(eda_df), 600), random_state=42).reset_index(drop=True)
stat_records = []

for _, row in tqdm(stat_sample.iterrows(), total=len(stat_sample)):
    img = Image.open(row['filepath']).convert('RGB')
    arr = np.array(img)
    h, w, _ = arr.shape
    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
    hsv = cv2.cvtColor(arr, cv2.COLOR_RGB2HSV)
    sobel = cv2.Sobel(gray, cv2.CV_64F, 1, 1, ksize=3)

    stat_records.append({
        'split': row['split'],
        'label': row['label'],
        'width': w,
        'height': h,
        'area': w * h,
        'aspect': w / h,
        'orientation': 'landscape' if w > h else ('portrait' if h > w else 'square'),
        'mean_r': arr[:, :, 0].mean(),
        'mean_g': arr[:, :, 1].mean(),
        'mean_b': arr[:, :, 2].mean(),
        'brightness_v': hsv[:, :, 2].mean(),
        'saturation_s': hsv[:, :, 1].mean(),
        'contrast_gray': gray.std(),
        'edge_strength': np.mean(np.abs(sobel)),
        'file_kb': os.path.getsize(row['filepath']) / 1024
    })

stat_df = pd.DataFrame(stat_records)
stat_df['resolution'] = stat_df['width'].astype(str) + 'x' + stat_df['height'].astype(str)

print(stat_df.head())
print('\nSample size used:', len(stat_df))

In [ ]:
# Viz 1: overall split distribution
plt.figure(figsize=(5,4))
sns.countplot(data=eda_df, x='split', order=['train','val','test'], palette='Set2')
plt.title('Viz 1: Images per split')
plt.xlabel('Split')
plt.ylabel('Count')
plt.show()

# Viz 2: overall class distribution (all splits)
plt.figure(figsize=(6,4))
sns.countplot(data=eda_df, x='label', order=sorted(eda_df['label'].unique()), palette='Set3')
plt.title('Viz 2: Images per class (all splits)')
plt.xlabel('Label')
plt.ylabel('Count')
plt.xticks(rotation=20)
plt.show()

In [ ]:
# Viz 3: stacked bar of class counts by split
counts = eda_df.groupby(['split','label']).size().unstack(fill_value=0)[sorted(eda_df['label'].unique())]
counts.loc[['train','val','test']].plot(kind='bar', stacked=True, figsize=(7,4), colormap='tab20')
plt.title('Viz 3: Class distribution per split (stacked)')
plt.xlabel('Split')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.legend(title='Label')
plt.show()

# Viz 4: grouped bar of class counts by split
counts.loc[['train','val','test']].plot(kind='bar', stacked=False, figsize=(7,4), colormap='Paired')
plt.title('Viz 4: Class distribution per split (grouped)')
plt.xlabel('Split')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.legend(title='Label')
plt.show()

In [ ]:
# Viz 5: heatmap of counts by split/label
plt.figure(figsize=(6,4))
sns.heatmap(counts.loc[['train','val','test']], annot=True, fmt='d', cmap='YlGnBu')
plt.title('Viz 5: Heatmap of class counts by split')
plt.xlabel('Label')
plt.ylabel('Split')
plt.show()

In [ ]:
# Viz 6: pie of label distribution (all splits)
plt.figure(figsize=(5,5))
eda_df['label'].value_counts().plot(kind='pie', autopct='%1.1f%%', colors=sns.color_palette('Set3'))
plt.ylabel('')
plt.title('Viz 6: Label share (all splits)')
plt.show()

# Viz 7: pie of split distribution
plt.figure(figsize=(4.5,4.5))
eda_df['split'].value_counts().reindex(['train','val','test']).plot(kind='pie', autopct='%1.1f%%', colors=sns.color_palette('Set2'))
plt.ylabel('')
plt.title('Viz 7: Split share')
plt.show()

In [ ]:
# Viz 8: histogram of image widths (sample)
plt.figure(figsize=(6,4))
sns.histplot(stat_df['width'], bins=30, color='steelblue', kde=True)
plt.title('Viz 8: Width distribution (sample)')
plt.xlabel('Width (px)')
plt.ylabel('Frequency')
plt.show()

# Viz 9: histogram of image heights (sample)
plt.figure(figsize=(6,4))
sns.histplot(stat_df['height'], bins=30, color='salmon', kde=True)
plt.title('Viz 9: Height distribution (sample)')
plt.xlabel('Height (px)')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Viz 10: hexbin of width vs height (sample)
plt.figure(figsize=(6,5))
plt.hexbin(stat_df['width'], stat_df['height'], gridsize=25, cmap='viridis')
plt.title('Viz 10: Width vs Height density')
plt.xlabel('Width (px)')
plt.ylabel('Height (px)')
cb = plt.colorbar()
cb.set_label('Count')
plt.show()

# Viz 11: aspect ratio histogram
plt.figure(figsize=(6,4))
sns.histplot(stat_df['aspect'], bins=30, color='mediumseagreen', kde=True)
plt.title('Viz 11: Aspect ratio (W/H) distribution')
plt.xlabel('Aspect ratio')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Viz 12: boxplot of width by label
plt.figure(figsize=(7,4))
sns.boxplot(data=stat_df, x='label', y='width', order=sorted(stat_df['label'].unique()), palette='Pastel1')
plt.title('Viz 12: Width by label (sample)')
plt.xlabel('Label')
plt.ylabel('Width (px)')
plt.xticks(rotation=20)
plt.show()

# Viz 13: boxplot of height by label
plt.figure(figsize=(7,4))
sns.boxplot(data=stat_df, x='label', y='height', order=sorted(stat_df['label'].unique()), palette='Pastel2')
plt.title('Viz 13: Height by label (sample)')
plt.xlabel('Label')
plt.ylabel('Height (px)')
plt.xticks(rotation=20)
plt.show()

# Viz 14: boxplot of aspect ratio by label
plt.figure(figsize=(7,4))
sns.boxplot(data=stat_df, x='label', y='aspect', order=sorted(stat_df['label'].unique()), palette='Set2')
plt.title('Viz 14: Aspect ratio by label (sample)')
plt.xlabel('Label')
plt.ylabel('Aspect ratio (W/H)')
plt.xticks(rotation=20)
plt.show()

In [ ]:
# Viz 15: orientation counts
plt.figure(figsize=(5,4))
sns.countplot(data=stat_df, x='orientation', order=['landscape','portrait','square'], palette='coolwarm')
plt.title('Viz 15: Orientation distribution (sample)')
plt.xlabel('Orientation')
plt.ylabel('Count')
plt.show()

# Viz 16: image area distribution (width*height)
plt.figure(figsize=(6,4))
sns.histplot(stat_df['area'], bins=30, color='mediumpurple', kde=True)
plt.title('Viz 16: Image area distribution (sample)')
plt.xlabel('Area (px^2)')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Viz 17: grayscale intensity histogram (sample)
plt.figure(figsize=(6,4))
gray_values = []
for path in stat_sample['filepath'][:200]:
    g = np.array(Image.open(path).convert('L')).flatten()
    gray_values.extend(g[np.random.choice(len(g), size=min(2000, len(g)), replace=False)])
sns.histplot(gray_values, bins=50, color='gray', kde=True)
plt.title('Viz 17: Grayscale intensity distribution (sample of pixels)')
plt.xlabel('Intensity (0-255)')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Viz 18: RGB channel histogram (sample)
plt.figure(figsize=(7,4))
channel_colors = ['red','green','blue']
for ch, color in enumerate(channel_colors):
    sns.histplot(stat_df[[f'mean_{c}' for c in ['r','g','b']][ch]], bins=30, color=color, label=f'{color} mean', alpha=0.5)
plt.title('Viz 18: Mean RGB values per image (sample)')
plt.xlabel('Channel mean (0-255)')
plt.ylabel('Frequency')
plt.legend()
plt.show()

In [ ]:
# Viz 19: mean RGB per class
rgb_means = stat_df.groupby('label')[['mean_r','mean_g','mean_b']].mean().reset_index()
rgb_melt = rgb_means.melt(id_vars='label', var_name='channel', value_name='mean_val')
plt.figure(figsize=(7,4))
sns.barplot(data=rgb_melt, x='label', y='mean_val', hue='channel', palette={'mean_r':'red','mean_g':'green','mean_b':'blue'})
plt.title('Viz 19: Mean RGB per class (sample)')
plt.xlabel('Label')
plt.ylabel('Channel mean (0-255)')
plt.xticks(rotation=20)
plt.legend(title='Channel')
plt.show()

In [ ]:
# Viz 20: brightness (V) by label
plt.figure(figsize=(7,4))
sns.boxplot(data=stat_df, x='label', y='brightness_v', order=sorted(stat_df['label'].unique()), palette='YlOrBr')
plt.title('Viz 20: Brightness (V channel) by label (sample)')
plt.xlabel('Label')
plt.ylabel('Brightness (0-255)')
plt.xticks(rotation=20)
plt.show()

# Viz 21: saturation (S) by label
plt.figure(figsize=(7,4))
sns.boxplot(data=stat_df, x='label', y='saturation_s', order=sorted(stat_df['label'].unique()), palette='PuBuGn')
plt.title('Viz 21: Saturation (S channel) by label (sample)')
plt.xlabel('Label')
plt.ylabel('Saturation (0-255)')
plt.xticks(rotation=20)
plt.show()

# Viz 22: contrast (std of gray) by label
plt.figure(figsize=(7,4))
sns.boxplot(data=stat_df, x='label', y='contrast_gray', order=sorted(stat_df['label'].unique()), palette='cool')
plt.title('Viz 22: Contrast (gray std) by label (sample)')
plt.xlabel('Label')
plt.ylabel('Std of gray')
plt.xticks(rotation=20)
plt.show()

# Viz 23: edge strength by label
plt.figure(figsize=(7,4))
sns.boxplot(data=stat_df, x='label', y='edge_strength', order=sorted(stat_df['label'].unique()), palette='rocket')
plt.title('Viz 23: Edge strength by label (sample)')
plt.xlabel('Label')
plt.ylabel('Mean abs Sobel response')
plt.xticks(rotation=20)
plt.show()

In [ ]:
# Viz 24: correlation heatmap of numeric stats
plt.figure(figsize=(8,6))
num_cols = ['width','height','area','aspect','mean_r','mean_g','mean_b','brightness_v','saturation_s','contrast_gray','edge_strength','file_kb']
corr = stat_df[num_cols].corr()
sns.heatmap(corr, cmap='coolwarm', center=0, annot=False)
plt.title('Viz 24: Correlation of image-level features (sample)')
plt.show()

# Viz 25: top resolutions by frequency (sample)
plt.figure(figsize=(7,4))
stat_df['resolution'].value_counts().head(10).plot(kind='bar', color='teal')
plt.title('Viz 25: Top 10 resolutions (sample)')
plt.xlabel('Resolution (WxH)')
plt.ylabel('Count')
plt.xticks(rotation=30)
plt.show()